In [ ]:
import torch
import copy
import sys
import os
import json
src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from functools import partial
from topic_watermark_processor import TopicWatermarkLogitsProcessor
from semantic_topic_extension import EmbeddingMapper
from model import (
    load_model, 
    load_sentence_model, 
    load_topic_model,
    detect,
)
from transformers import LogitsProcessorList, AutoTokenizer
from watermark.auto_watermark import AutoWatermark
from watermark.transformers_config import TransformersConfig

from attacks import paraphrase
import matplotlib.pyplot as plt
from datasets import load_dataset
import numpy as np
from sklearn.metrics import roc_curve, f1_score, auc
import nltk
nltk.download('punkt_tab')


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

args = {
    # 'model_name_or_path': 'facebook/opt-2.7b',
    # 'model_name_or_path': 'facebook/opt-1.3b',
#     'model_name_or_path': 'facebook/opt-125m', 
    # 'model_name_or_path': 'meta-llama/Llama-3.2-1B',
#     'model_name_or_path': 'facebook/opt-6.7b',
    'model_name_or_path': 'google/gemma-7b',
    'load_fp16' : True,
    'prompt_max_length': None, 
    'max_new_tokens': 200, 
    'generation_seed': 123, 
    'use_sampling': True, 
    'n_beams': 1, 
    'sampling_temp': 0.7, 
    'use_gpu': True, 
    'seeding_scheme': 'simple_1', 
    'delta': 3.0, 
    'ignore_repeated_bigrams': False, 
    'detection_z_threshold': 4.75, 
    'select_green_tokens': True,
    'skip_model_load': False,
    'seed_separately': True,
    'topic_token_mapping': {},
    'granularity': 'kmeans',
}

In [ ]:
c4_dataset = load_dataset('json', data_files='./c4/c4.json', split='train[:1000]')
c4_iter = iter(c4_dataset)
n = 500  # how many prompts to sample
inputs_list = []
for _ in range(n):
    entry = next(c4_iter)
    text = entry["text"]
    # first 100 words
    shortened_text = " ".join(text.split()[:100])
    inputs_list.append(shortened_text)

In [ ]:
topic_list = ["animals", "technology", "sports", "medicine"]

model, tokenizer = load_model(args)
print("Model loaded")

In [ ]:
sentence_model = load_sentence_model()
print("Sentence model loaded")
embedding_mapper = EmbeddingMapper(tokenizer, sentence_model)
total_tokens, vocab_embeddings = embedding_mapper.get_model_vocab_embeddings()
topic_embeddings = embedding_mapper.get_defined_topic_list_embeddings(topic_list)
topic_token_mapping = embedding_mapper.map_tokens_to_topics(total_tokens, vocab_embeddings, topic_list, topic_embeddings)
args['topic_token_mapping'] = topic_token_mapping

In [ ]:
def generate_no_watermark(prompt, args, model=None, tokenizer=None):
    gen_kwargs = {
        'max_new_tokens': args['max_new_tokens'],
        'min_new_tokens': args['min_new_tokens']
    }

    if args['use_sampling']:
        gen_kwargs.update(dict(
            do_sample=True, 
            top_k=0,
            temperature=args['sampling_temp'],
        ))
    else:
        gen_kwargs.update(dict(
            num_beams=args['n_beams']
        ))

    # generate without the watermark 
    generate_without_watermark = partial(
        model.generate,
        **gen_kwargs
    ) 

    if args['prompt_max_length']:
        pass
    elif hasattr(model.config,"max_position_embeddings"):
        args['prompt_max_length'] = model.config.max_position_embeddings - args['max_new_tokens']
    else:
        args['prompt_max_length'] = 2048 - args['max_new_tokens']

    tokenized_input = tokenizer(
        prompt, 
        return_tensors="pt", 
        add_special_tokens=True, 
        truncation=True, 
        max_length=args['prompt_max_length']
    ).to(device)

    torch.manual_seed(args['generation_seed'])

    output_without_watermark = generate_without_watermark(**tokenized_input)

    if args['decoder']:
        # need to isolate the newly generated tokens
        output_without_watermark = output_without_watermark[:,tokenized_input["input_ids"].shape[-1]:]

    decoded_output_without_watermark = tokenizer.batch_decode(output_without_watermark, skip_special_tokens=True)[0]

    return decoded_output_without_watermark

def generate_TBW(prompt, detected_topics, args, model=None, tokenizer=None, sentence_model=None):

    detected_topic = ''
    for topic in detected_topics:
        if topic.lower() in args['topic_token_mapping']:
            detected_topic = topic.lower()
            print(f"Topic detected in one to one mapping: {detected_topic}")
            break
    if detected_topic == '':
        embedding_mapper = EmbeddingMapper(tokenizer, sentence_model)

        if args['granularity'] == 'kmeans':
            print(f"Mapping topic with granularity Kmeans.")
            detected_topic, _ = embedding_mapper.kmeans_detected_topics_to_embeddings(
                list(args['topic_token_mapping'].keys()), 
                detected_topics,
            )
        else:
            print(f"Mapping topic with granularity averaging")
            detected_topic, _ = embedding_mapper.detected_topics_to_embeddings(
                list(args['topic_token_mapping'].keys()), 
                detected_topics
            )

    print(f"Detected topic: {detected_topic}")
    watermark_processor = TopicWatermarkLogitsProcessor(
        vocab=list(tokenizer.get_vocab().values()),
        delta=args['delta'],
        seeding_scheme=args['seeding_scheme'],
        select_green_tokens=args['select_green_tokens'],
        topic_token_mapping=args['topic_token_mapping'],
        detected_topic=detected_topic,
    )
  
    gen_kwargs = {
        'max_new_tokens': args['max_new_tokens'],
        'min_new_tokens': args['min_new_tokens']
    }

    if args['use_sampling']:
        gen_kwargs.update(dict(
            do_sample=True, 
            top_k=0,
            temperature=args['sampling_temp'],
            repetition_penalty=1.1,
        ))
    else:
        gen_kwargs.update(dict(
            num_beams=args['n_beams']
        ))

    generate_with_watermark = partial(
        model.generate,
        logits_processor=LogitsProcessorList([watermark_processor]), 
        **gen_kwargs
    )

    if args['prompt_max_length']:
        pass
    elif hasattr(model.config,"max_position_embeddings"):
        args['prompt_max_length'] = model.config.max_position_embeddings - args['max_new_tokens']
    else:
        args['prompt_max_length'] = 2048 - args['max_new_tokens']

    tokenized_input = tokenizer(
        prompt, 
        return_tensors="pt", 
        add_special_tokens=True, 
        truncation=True, 
        max_length=args['prompt_max_length']
    ).to(device)

    output_with_watermark = generate_with_watermark(**tokenized_input)

    if args['decoder']:
        # need to isolate the newly generated tokens
        output_with_watermark = output_with_watermark[:,tokenized_input["input_ids"].shape[-1]:]

    decoded_output_with_watermark = tokenizer.batch_decode(output_with_watermark, skip_special_tokens=True)[0]

    return decoded_output_with_watermark

def generate_KGW(prompt, transformers_config):
    myWatermark = AutoWatermark.load('KGW', transformers_config=transformers_config)
    watermarked_text = myWatermark.generate_watermarked_text(prompt)
    return watermarked_text

def generate_DIP(prompt, transformers_config):
    myWatermark = AutoWatermark.load('DIP', transformers_config=transformers_config)
    watermarked_text = myWatermark.generate_watermarked_text(prompt)
    return watermarked_text

def generate_Unigram(prompt, transformers_config):
    myWatermark = AutoWatermark.load('Unigram', transformers_config=transformers_config)
    watermarked_text = myWatermark.generate_watermarked_text(prompt)
    return watermarked_text

def generate_SynthID(prompt, transformers_config):
    myWatermark = AutoWatermark.load('SynthID', transformers_config=transformers_config)
    watermarked_text = myWatermark.generate_watermarked_text(prompt)
    return watermarked_text

def generate_EXP(prompt, transformers_config):
    myWatermark = AutoWatermark.load('EXP', transformers_config=transformers_config)
    watermarked_text = myWatermark.generate_watermarked_text(prompt)
    return watermarked_text

def generate_EXPEdit(prompt, transformers_config):
    myWatermark = AutoWatermark.load('EXPEdit', transformers_config=transformers_config)
    watermarked_text = myWatermark.generate_watermarked_text(prompt)
    return watermarked_text

def generate_ITSEdit(prompt, transformers_config):
    myWatermark = AutoWatermark.load('ITSEdit', transformers_config=transformers_config)
    watermarked_text = myWatermark.generate_watermarked_text(prompt)
    return watermarked_text

def generate_SIR(prompt, transformers_config):
    myWatermark = AutoWatermark.load('SIR', transformers_config=transformers_config)
    watermarked_text = myWatermark.generate_watermarked_text(prompt)
    return watermarked_text

In [ ]:
schemes = {
    "TBW": generate_TBW,
    "KGW": generate_KGW,
    "DIP": generate_DIP,
    "Unigram": generate_Unigram,
    "SynthID": generate_SynthID,
    "SIR": generate_SIR,
}

warmup_length = 10
transformers_config = TransformersConfig(
    model=model,
    tokenizer=tokenizer,
    vocab_size=256000,
    device=device,
    max_new_tokens=warmup_length,
    do_sample=True,
    no_repeat_ngram_size=4
)
for scheme, func in schemes.items():
    input_text = inputs_list[0]
    args_warmup = copy.deepcopy(args)
    args_warmup['max_new_tokens'] = warmup_length
    args_warmup['min_new_tokens'] = warmup_length
    generated_text = generate_no_watermark(input_text, args_warmup, model=model, tokenizer=tokenizer)

    print(f"Done generating for scheme {scheme}")
print("Warmup done")


gen_new_tokens = 200
transformers_config = TransformersConfig(
    model=model,
    tokenizer=tokenizer,
    vocab_size=256000,
    device=device,
    max_new_tokens=gen_new_tokens+5,
    min_new_tokens=gen_new_tokens-5,
    do_sample=True,
    no_repeat_ngram_size=4
)
args_gen = copy.deepcopy(args)
args_gen['max_new_tokens'] = gen_new_tokens+5
args_gen['min_new_tokens'] = gen_new_tokens-5
generated_text_per_scheme_no_watermark = {}

generated_text_per_scheme = {}
for scheme, func in schemes.items():
    
    texts_w = []
    count = 0

    for input_text in inputs_list:
        if scheme == 'TBW':
            detected_topics = load_topic_model(input_text)
            generated_text = func(input_text, detected_topics, args_gen, model=model, tokenizer=tokenizer, sentence_model=sentence_model)
            texts_w.append(generated_text)
        else:
            generated_text = func(input_text, transformers_config)
            if generated_text.startswith(input_text):
                generated_text = generated_text[len(input_text):]
            texts_w.append(generated_text)
        print(f"Text {count} for {scheme}")
        print(generated_text)
        count +=1
        
    print(f"Done generating for scheme {scheme}")
    generated_text_per_scheme[scheme] = texts_w
        

In [ ]:
pegasus_paraphraser_texts = {}
pegasus_paraphraser = paraphrase.ParaphrasingAttack(model_type="pegasus")
for scheme, generated_texts in generated_text_per_scheme.items():
    print(scheme)
    pegasus_paraphraser_texts[scheme] = pegasus_paraphraser.paraphrase_processor(
                                                            watermarked_text_list=generated_texts,
                                                            num_sequences=1, 
                                                            num_beams=3
                                                        )


In [ ]:
dipper_paraphraser_texts = {}
dipper_paraphraser = paraphrase.ParaphrasingAttack(model_type="dipper")
for scheme, generated_texts in generated_text_per_scheme.items():
    print(scheme)
    if scheme == 'TBW':
        dipper_paraphraser_texts[scheme] = dipper_paraphraser.paraphrase_processor(
                                                                watermarked_text_list=generated_texts,
                                                                lexical=20,
                                                                order=40
                                                            )

In [ ]:
def detect_KGW(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('KGW', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_DIP(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('DIP', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_Unigram(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('Unigram', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_SynthID(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('SynthID', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_EXP(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('EXP', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_EXPEdit(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('EXPEdit', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_ITSEdit(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('ITSEdit', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_SIR(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('SIR', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

In [ ]:
schemes_detection = {
    "TBW": detect,
    "KGW": detect_KGW,
    "DIP": detect_DIP,
    "Unigram": detect_Unigram,
    "SynthID": detect_SynthID,
    "SIR": detect_SIR
}

#     "EXP": detect_EXP,
#     "EXPEdit": detect_EXPEdit,
#     "ITSEdit": detect_ITSEdit,


# detected_results_no_watermark = {}
# for scheme, texts in generated_text_per_scheme_no_watermark.items():
#     print(scheme)
#     detection_function = schemes_detection.get(scheme)
#     detection_results = []
#     if scheme == 'TBW':
#         for idx, text in enumerate(texts):
#             result_formatted = detection_function(inputs_list[idx], text, args_gen, device=device, tokenizer=tokenizer, sentence_model=sentence_model)
#             result = {
#                 'is_watermarked': result_formatted[6][1] == 'Watermarked',
#                 'score': float(result_formatted[3][1])
#             }
#             print(result)
#             detection_results.append(result)
#     else:
#         for text in texts:
#             result = detection_function(text, transformers_config)
#             print(result)
#             detection_results.append(result)
#     detected_results_no_watermark[scheme] = detection_results

detected_results = {}
for scheme, texts in generated_text_per_scheme.items():
    print(scheme)
    detection_function = schemes_detection.get(scheme)
    detection_results = []
    if scheme == 'TBW':
        for idx, text in enumerate(texts):
            result_formatted = detection_function(inputs_list[idx], text, args_gen, device=device, tokenizer=tokenizer, sentence_model=sentence_model)
            result = {
                'is_watermarked': result_formatted[6][1] == 'Watermarked',
                'score': float(result_formatted[3][1])
            }
            print(result)
            detection_results.append(result)
    else:
        for text in texts:
            result = detection_function(text, transformers_config)
            print(result)
            detection_results.append(result)
    detected_results[scheme] = detection_results

In [ ]:
def convert_numpy_types(obj):
    if isinstance(obj, np.bool_):
        return bool(obj)
    elif isinstance(obj, np.ndarray): 
        return obj.tolist()
    elif isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    elif isinstance(obj, dict): 
        return {key: convert_numpy_types(value) for key, value in obj.items()}
    elif isinstance(obj, list): 
        return [convert_numpy_types(element) for element in obj]
    else:
        return obj

In [ ]:
combined_results_no_attack = {}

schemes_detection = {
    "TBW": detect,
    "KGW": detect_KGW,
    "DIP": detect_DIP,
    "Unigram": detect_Unigram,
    "SynthID": detect_SynthID,
    "SIR": detect_SIR
}


for scheme in schemes_detection.keys():
    nonwm_results = detected_results_no_watermark.get(scheme, [])
    wm_results = detected_results.get(scheme, [])

    combined_list = []
    for result in nonwm_results:
        combined_list.append(
            (result['score'], 0)  
        )
    
    for result in wm_results:
        combined_list.append(
            (result['score'], 1) 
        )
    
    combined_results_no_attack[scheme] = combined_list


for key, value in combined_results_no_attack.items():
    print(f"Key: {key}, Value: {len(value)}")

In [ ]:
def get_tpr_f1_at_fpr_levels(fpr, tpr, thresholds, scores, labels, fpr_levels=[0.01, 0.1]):
    results = {}
    fpr_tpr_thresholds = list(zip(fpr, tpr, thresholds))
    fpr_tpr_thresholds.sort(key=lambda x: x[0])  # sort by fpr ascending
    fpr_sorted, tpr_sorted, thresh_sorted = zip(*fpr_tpr_thresholds)
    
    fpr_sorted = np.array(fpr_sorted)
    tpr_sorted = np.array(tpr_sorted)
    thresh_sorted = np.array(thresh_sorted)
    
    for level in fpr_levels:
        idx_candidates = np.where(fpr_sorted >= level)[0]
        if len(idx_candidates) == 0:
            idx = len(fpr_sorted) - 1
        else:
            idx = idx_candidates[0]
        
        chosen_threshold = thresh_sorted[idx]
        chosen_tpr = tpr_sorted[idx]
        
        preds = (scores >= chosen_threshold).astype(int)
        chosen_f1 = f1_score(labels, preds)
        
        results[level] = {
            "threshold": chosen_threshold,
            "tpr": chosen_tpr,
            "f1": chosen_f1
        }
    
    return results


In [ ]:
results_metrics = {}

for scheme, score_label_list in combined_results_no_attack.items():
    scores = np.array([sl[0] for sl in score_label_list])
    labels = np.array([sl[1] for sl in score_label_list])
    
    fpr, tpr, thresholds = roc_curve(labels, scores)
    roc_auc = auc(fpr, tpr)
    
    best_f1 = 0
    best_threshold = None
    for th in thresholds:
        preds = (scores >= th).astype(int)
        f1 = f1_score(labels, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = th
    
    
    fpr_levels_results = get_tpr_f1_at_fpr_levels(fpr, tpr, thresholds, scores, labels, fpr_levels=[0.01, 0.1])
    
    results_metrics[scheme] = {
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
        "roc_auc": roc_auc,
        "best_f1": best_f1,
        "best_threshold": best_threshold, 
        "fpr_levels_results": fpr_levels_results
    }

print(results_metrics)


In [ ]:
schemes_detection = {
    "TBW": detect,
    "KGW": detect_KGW,
    "DIP": detect_DIP,
    "Unigram": detect_Unigram,
    "SynthID": detect_SynthID,
    "EXP": detect_EXP,
    "EXPEdit": detect_EXPEdit,
    "ITSEdit": detect_ITSEdit,
    "SIR": detect_SIR,
}


detected_results_no_watermark_dipper = {}
for scheme, texts in dipper_paraphraser_texts_no_watermark.items():
    print(scheme)
    detection_function = schemes_detection.get(scheme)
    detection_results = []
    if scheme == 'TBW':
        for idx, text in enumerate(texts):
            result_formatted = detection_function(inputs_list[idx], text, args_gen, device=device, tokenizer=tokenizer, sentence_model=sentence_model)
            result = {
                'is_watermarked': result_formatted[6][1] == 'Watermarked',
                'score': float(result_formatted[3][1])
            }
            detection_results.append(result)
    else:
        for text in texts:
            result = detection_function(text, transformers_config)
            detection_results.append(result)
    detected_results_no_watermark_dipper[scheme] = detection_results

detected_results_dipper = {}
for scheme, texts in dipper_paraphraser_texts.items():
    print(scheme)
    detection_function = schemes_detection.get(scheme)
    detection_results = []
    if scheme == 'TBW':
        for idx, text in enumerate(texts):
            result_formatted = detection_function(inputs_list[idx], text, args_gen, device=device, tokenizer=tokenizer, sentence_model=sentence_model)
            result = {
                'is_watermarked': result_formatted[6][1] == 'Watermarked',
                'score': float(result_formatted[3][1])
            }
            detection_results.append(result)
    else:
        for text in texts:
            result = detection_function(text, transformers_config)
            detection_results.append(result)
    detected_results_dipper[scheme] = detection_results

In [ ]:
combined_results_dipper = {}

for scheme in schemes_detection.keys():
    nonwm_results = detected_results_no_watermark_dipper.get(scheme, [])
    wm_results = detected_results_dipper.get(scheme, [])

    combined_list = []
    for result in nonwm_results:
        combined_list.append(
            (result['score'], 0)  
        )
    
    for result in wm_results:
        combined_list.append(
            (result['score'], 1) 
        )
    
    combined_results_dipper[scheme] = combined_list


for key, value in combined_results_dipper.items():
    print(f"Key: {key}, Value: {value}")

In [ ]:
results_metrics_dipper = {}

for scheme, score_label_list in combined_results_dipper.items():
    scores = np.array([sl[0] for sl in score_label_list])
    labels = np.array([sl[1] for sl in score_label_list])
    
    fpr, tpr, thresholds = roc_curve(labels, scores)
    roc_auc = auc(fpr, tpr)
    
    best_f1 = 0
    best_threshold = None
    for th in thresholds:
        preds = (scores >= th).astype(int)
        f1 = f1_score(labels, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = th
    
    results_metrics_dipper[scheme] = {
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
        "roc_auc": roc_auc,
        "best_f1": best_f1,
        "best_threshold": best_threshold
    }

print(results_metrics_dipper)

In [ ]:
schemes_detection = {
    "TBW": detect,
    "KGW": detect_KGW,
    "DIP": detect_DIP,
    "Unigram": detect_Unigram,
    "SynthID": detect_SynthID,
    "EXP": detect_EXP,
    "EXPEdit": detect_EXPEdit,
    "ITSEdit": detect_ITSEdit,
    "SIR": detect_SIR,
}


detected_results_no_watermark_pegasus = {}
for scheme, texts in pegasus_paraphraser_texts_no_watermark.items():
    print(scheme)
    detection_function = schemes_detection.get(scheme)
    detection_results = []
    if scheme == 'TBW':
        for idx, text in enumerate(texts):
            result_formatted = detection_function(inputs_list[idx], text, args_gen, device=device, tokenizer=tokenizer, sentence_model=sentence_model)
            result = {
                'is_watermarked': result_formatted[6][1] == 'Watermarked',
                'score': float(result_formatted[3][1])
            }
            detection_results.append(result)
    else:
        for text in texts:
            result = detection_function(text, transformers_config)
            detection_results.append(result)
    detected_results_no_watermark_pegasus[scheme] = detection_results

detected_results_pegasus = {}
for scheme, texts in pegasus_paraphraser_texts.items():
    print(scheme)
    detection_function = schemes_detection.get(scheme)
    detection_results = []
    if scheme == 'TBW':
        for idx, text in enumerate(texts):
            result_formatted = detection_function(inputs_list[idx], text, args_gen, device=device, tokenizer=tokenizer, sentence_model=sentence_model)
            result = {
                'is_watermarked': result_formatted[6][1] == 'Watermarked',
                'score': float(result_formatted[3][1])
            }
            detection_results.append(result)
    else:
        for text in texts:
            result = detection_function(text, transformers_config)
            detection_results.append(result)
    detected_results_pegasus[scheme] = detection_results

In [ ]:
combined_results_pegasus = {}

for scheme in schemes_detection.keys():
    nonwm_results = detected_results_no_watermark_pegasus.get(scheme, [])
    wm_results = detected_results_pegasus.get(scheme, [])

    combined_list = []
    for result in nonwm_results:
        combined_list.append(
            (result['score'], 0)  
        )
    
    for result in wm_results:
        combined_list.append(
            (result['score'], 1) 
        )
    
    combined_results_pegasus[scheme] = combined_list


for key, value in combined_results_pegasus.items():
    print(f"Key: {key}, Value: {value}")

In [ ]:
results_metrics_pegasus = {}

for scheme, score_label_list in combined_results_pegasus.items():
    scores = np.array([sl[0] for sl in score_label_list])
    labels = np.array([sl[1] for sl in score_label_list])
    
    fpr, tpr, thresholds = roc_curve(labels, scores)
    roc_auc = auc(fpr, tpr)
    
    best_f1 = 0
    best_threshold = None
    for th in thresholds:
        preds = (scores >= th).astype(int)
        f1 = f1_score(labels, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = th
    
    results_metrics_pegasus[scheme] = {
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
        "roc_auc": roc_auc,
        "best_f1": best_f1,
        "best_threshold": best_threshold
    }

print(results_metrics_pegasus)